In [1]:
   import pandas as pd
   df = pd.read_parquet("data/raw/energy_hourly.parquet")
   print(df.shape)
   print(df["series_key"].value_counts())
   df[df["series_key"] == "day_ahead_price"]["value"].describe()

(217533, 8)
series_key
total_load            15539
residual_load         15539
wind_onshore          15539
solar                 15539
wind_offshore         15539
biomass               15539
hard_coal             15539
hydro                 15539
other_renewable       15539
lignite               15539
pumped_storage        15539
natural_gas           15539
other_conventional    15539
day_ahead_price       15526
Name: count, dtype: int64


count    15526.000000
mean        98.167676
std         59.709349
min       -499.000000
25%         74.090000
50%         99.860000
75%        128.497500
max        697.310000
Name: value, dtype: float64

In [2]:
price = df[df["series_key"] == "day_ahead_price"].copy()
price["ts_local"] = price["ts_utc"].dt.tz_convert("Europe/Berlin")

# 1. The 5 cheapest and 5 most expensive hours
print("Cheapest hours:")
print(price.nsmallest(5, "value")[["ts_local", "value"]])
print("\nMost expensive hours:")
print(price.nlargest(5, "value")[["ts_local", "value"]])

# 2. How many negative-price hours?
print("\nNegative-price hours:", (price["value"] < 0).sum())

# 3. Which 13 hours are missing?
load_hours = set(df[df["series_key"] == "total_load"]["ts_utc"])
missing = sorted(load_hours - set(price["ts_utc"]))
print("\nMissing price hours:")
for t in missing:
    print(t.tz_convert("Europe/Berlin"))

Cheapest hours:
                       ts_local   value
11651 2026-05-01 13:00:00+02:00 -499.00
11652 2026-05-01 14:00:00+02:00 -474.97
11531 2026-04-26 13:00:00+02:00 -413.77
11532 2026-04-26 14:00:00+02:00 -413.25
11650 2026-05-01 12:00:00+02:00 -318.58

Most expensive hours:
                       ts_local   value
14921 2026-09-14 19:00:00+02:00  697.31
12954 2026-06-24 20:00:00+02:00  665.82
12955 2026-06-24 21:00:00+02:00  660.16
15113 2026-09-22 19:00:00+02:00  596.38
472   2025-01-20 17:00:00+01:00  583.40

Negative-price hours: 1033

Missing price hours:
2026-10-10 00:00:00+02:00
2026-10-10 01:00:00+02:00
2026-10-10 02:00:00+02:00
2026-10-10 03:00:00+02:00
2026-10-10 04:00:00+02:00
2026-10-10 05:00:00+02:00
2026-10-10 06:00:00+02:00
2026-10-10 07:00:00+02:00
2026-10-10 08:00:00+02:00
2026-10-10 09:00:00+02:00
2026-10-10 10:00:00+02:00
2026-10-10 11:00:00+02:00
2026-10-10 12:00:00+02:00


In [3]:
wide = df.pivot_table(index="ts_utc", columns="series_key", values="value")
renew = ["solar", "wind_onshore", "wind_offshore", "biomass", "hydro", "other_renewable"]
conv = ["lignite", "hard_coal", "natural_gas", "pumped_storage", "other_conventional"]
wide["renewable_share"] = wide[renew].sum(axis=1) / wide[renew + conv].sum(axis=1)
wide = wide.dropna(subset=["day_ahead_price"])

wide["share_band"] = pd.cut(wide["renewable_share"], bins=[0, .3, .5, .7, .9, 1],
                            labels=["<30%", "30-50%", "50-70%", "70-90%", ">90%"])
wide.groupby("share_band", observed=True)["day_ahead_price"].agg(["count", "mean"]).round(1)

,count,mean
share_band,,
<30%,1392,161.6
30-50%,4548,131.8
50-70%,4722,106.7
70-90%,4591,43.9
>90%,273,-20.6


# Findings - Week 1
## Initial findings (data: SMARD, Jan 2025 – Oct 2026, 15,526 hourly prices)

**1. More renewables = lower prices**
| Renewable share | Hours | Avg price (€/MWh) |
|---|---|---|
| < 30% | 1,392 | 161.6 |
| 30–50% | 4,548 | 131.8 |
| 50–70% | 4,722 | 106.7 |
| 70–90% | 4,591 | 43.9 |
| > 90% | 273 | −20.6 |

Hours with 70–90% renewables were 73% cheaper than hours below 30%.
Above 90%, the average price was negative (merit-order effect).

**2. Negative prices are common**
1,033 hours (6.7%) had negative prices. The lowest was −€499/MWh on
1 May 2026 at 13:00, a public holiday with strong solar and low demand.
The next lowest was Sunday 26 April 2026, at the same midday hours.

**3. Evenings are the most expensive hours**
All of the 5 most expensive hours fell between 17:00 and 21:00
(highest: €697/MWh on 14 Sep 2026, 19:00), after solar generation
drops but demand stays high.

**4. Data quality checks**
- Nuclear = 0 rows: correct, Germany's last plants closed in April 2023.
- 13 missing price hours: all today (10 Oct 2026), not yet published
  by SMARD; the daily refresh fills them in.

**Caveat:** this shows correlation, not sole causation. Time of day,
season, demand and gas prices also affect prices.

**Next:** check price by hour of day and weekday vs. weekend.